# FloodWatch Upgraded Architecture — EfficientNet-B4 + V-FloodNet Weights
**Google Colab notebook. Run cells in order (top to bottom).**

| Cell | What it does |
|------|--------------|
| 1 | Config — edit these values |
| 2 | Create all folders |
| 3 | Install packages |
| 4 | Download V-FloodNet pre-trained weights |
| 5 | Gemini auto-label all images → labels.csv |
| 6 | Split: 400 train / rest test |
| 7 | Build FloodNetV2 (B4 encoder + 3 heads) |
| 8 | Define augmentations + dataset |
| 9 | Multi-task training loop |
| 10 | Evaluate on test set |
| 11 | Deep metrics (optional) |
| 12 | Download results ZIP |


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 1 — CONFIGURATION  (edit these before running)
# ─────────────────────────────────────────────────────────────────

GEMINI_API_KEY = ''          # paste your Gemini API key here

TRAIN_SPLIT     = 400        # images used for training
EPOCHS          = 30
BATCH_SIZE      = 16
IMAGE_SIZE      = 416        # must be divisible by 32
LEARNING_RATE   = 0.0001
ENCODER_NAME    = 'efficientnet-b4'
ENCODER_WEIGHTS = 'imagenet'  # fall-back if V-FloodNet download fails

# Multi-task loss weights  (must sum to 1.0)
W_SEG   = 0.4
W_RISK  = 0.4
W_DEPTH = 0.2

# Risk classes
RISK_CLASSES = ['no_flood', 'low', 'moderate', 'high', 'critical']
NUM_CLASSES  = len(RISK_CLASSES)

# Paths (do NOT change unless you know what you are doing)
IMG_DIR      = '/content/images'
LABEL_PATH   = '/content/labels/labels.csv'
CKPT_DIR     = '/content/checkpoints'
PRETRAIN_DIR = '/content/pretrained'
OUT_DIR      = '/content/output'

import os, random
random.seed(42)
print('Config loaded')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 2 — CREATE ALL FOLDERS
# ─────────────────────────────────────────────────────────────────

FOLDERS = [
    '/content/images',
    '/content/labels',
    '/content/checkpoints',
    '/content/pretrained',
    '/content/output',
    '/content/output/charts',
    '/content/output/metrics',
    '/content/output/grids',
]
for f in FOLDERS:
    os.makedirs(f, exist_ok=True)
    print(f'  ✔ {f}')
print('\nAll folders ready.')
print(f'\n>>> Upload your images to  {IMG_DIR}/  then continue.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 3 — INSTALL PACKAGES
#  NOTE: do NOT re-install torch/torchvision — Colab has them
# ─────────────────────────────────────────────────────────────────

import subprocess, sys

pkgs = [
    'segmentation-models-pytorch',
    'timm',
    'google-generativeai',
    'albumentations',
    'scikit-learn',
    'matplotlib',
    'pillow',
]
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=True)
    print(f'  ✔ {p}')

import torch
print(f'\nPyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 4 — DOWNLOAD V-FloodNet PRE-TRAINED WEIGHTS
#  Falls back to ImageNet if download fails (safe fallback)
# ─────────────────────────────────────────────────────────────────

import urllib.request, pathlib, time

PRETRAIN_PATH  = f'{PRETRAIN_DIR}/link_efficientb4_model.pth'
HF_URL = (
    'https://huggingface.co/xmlyqing00/V-FloodNet/resolve/main'
    '/records/link_efficientb4_model.pth?download=true'
)

USE_VFLOODNET = False

if not pathlib.Path(PRETRAIN_PATH).exists():
    print('Downloading V-FloodNet checkpoint...')
    try:
        def _progress(count, block, total):
            if total > 0:
                pct = min(100, int(count * block * 100 / total))
                print(f'\r  {pct}%', end='', flush=True)
        urllib.request.urlretrieve(HF_URL, PRETRAIN_PATH, reporthook=_progress)
        size_mb = pathlib.Path(PRETRAIN_PATH).stat().st_size / 1e6
        if size_mb > 5:                        # sanity check — real ckpt > 50 MB
            USE_VFLOODNET = True
            print(f'\n  ✔ Downloaded {size_mb:.1f} MB')
        else:
            print(f'\n  ✗ Downloaded file too small ({size_mb:.1f} MB) — using ImageNet weights')
            pathlib.Path(PRETRAIN_PATH).unlink(missing_ok=True)
    except Exception as e:
        print(f'\n  ✗ Download failed: {e} — using ImageNet weights')
        pathlib.Path(PRETRAIN_PATH).unlink(missing_ok=True)
else:
    USE_VFLOODNET = True
    print(f'  ✔ Checkpoint already present at {PRETRAIN_PATH}')

print(f'Use V-FloodNet weights: {USE_VFLOODNET}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 5 — GEMINI AUTO-LABEL ALL IMAGES  → labels.csv
#  Rate limit: 15 req/min free tier → DELAY_SEC = 4 keeps us safe
# ─────────────────────────────────────────────────────────────────

import csv, json, re, time, pathlib
import google.generativeai as genai
from PIL import Image

DELAY_SEC = 4          # seconds between requests (free tier: 15 req/min)
RESUME    = True       # skip images already in labels.csv

if not GEMINI_API_KEY:
    raise ValueError('Set GEMINI_API_KEY in Cell 1 before running this cell')

genai.configure(api_key=GEMINI_API_KEY)
model_g = genai.GenerativeModel('gemini-1.5-flash')

PROMPT = """
Analyse this image for flood conditions. Return ONLY valid JSON (no markdown):
{"flood_detected": true/false,
 "water_depth_cm": <integer 0-400>,
 "risk_level": "no_flood|low|moderate|high|critical",
 "water_coverage_pct": <integer 0-100>}
"""

image_files = sorted([
    p for p in pathlib.Path(IMG_DIR).iterdir()
    if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')
])

if not image_files:
    raise RuntimeError(f'No images found in {IMG_DIR} — upload images first (Cell 2 prints the path)')

# Load already-labelled filenames if resuming
done = set()
if RESUME and pathlib.Path(LABEL_PATH).exists():
    with open(LABEL_PATH) as f:
        for row in csv.DictReader(f):
            done.add(row['filename'])
    print(f'Resuming: {len(done)} already labelled, {len(image_files)-len(done)} remaining')

def parse_gemini(text):
    text = re.sub(r'```(?:json)?|```', '', text).strip()
    return json.loads(text)

FIELDNAMES = ['filename','flood_detected','water_depth_cm','risk_level','water_coverage_pct']
mode = 'a' if done else 'w'
with open(LABEL_PATH, mode, newline='') as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    if mode == 'w':
        writer.writeheader()

    errors = []
    for i, img_path in enumerate(image_files):
        fname = img_path.name
        if fname in done:
            continue
        try:
            img = Image.open(img_path).convert('RGB')
            response = model_g.generate_content([PROMPT, img])
            data     = parse_gemini(response.text)
            writer.writerow({
                'filename':         fname,
                'flood_detected':   str(data.get('flood_detected', False)).lower(),
                'water_depth_cm':   int(data.get('water_depth_cm', 0)),
                'risk_level':       data.get('risk_level', 'no_flood'),
                'water_coverage_pct': int(data.get('water_coverage_pct', 0)),
            })
            f.flush()
            done.add(fname)
            if (i + 1) % 20 == 0:
                print(f'  {len(done)}/{len(image_files)} labelled...')
        except Exception as e:
            errors.append((fname, str(e)))
            print(f'  ✗ {fname}: {e}')
        time.sleep(DELAY_SEC)

print(f'\nLabelling complete: {len(done)} labelled, {len(errors)} errors')
if errors:
    print('Errors:', errors[:5])

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 6 — SPLIT: 400 TRAIN / REST TEST
# ─────────────────────────────────────────────────────────────────

import csv, random, collections

with open(LABEL_PATH) as f:
    all_rows = list(csv.DictReader(f))

# Filter out images that don't exist on disk
all_rows = [
    r for r in all_rows
    if (pathlib.Path(IMG_DIR) / r['filename']).exists()
]
random.shuffle(all_rows)

n_train = min(TRAIN_SPLIT, len(all_rows))
train_rows = all_rows[:n_train]
test_rows  = all_rows[n_train:]

def class_dist(rows):
    c = collections.Counter(r['risk_level'] for r in rows)
    return dict(c)

print(f'Total labelled: {len(all_rows)}')
print(f'Train: {len(train_rows)}  |  Test: {len(test_rows)}')
print(f'Train class dist: {class_dist(train_rows)}')
print(f'Test  class dist: {class_dist(test_rows)}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 7 — BUILD FloodNetV2
#  EfficientNet-B4 encoder → LinkNet decoder (water mask)
#                           → risk head (5-class)
#                           → depth head (regression)
#
#  FIX: encoder channels detected dynamically (not hardcoded)
# ─────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

class FloodNetV2(nn.Module):
    def __init__(self, num_classes=5, image_size=416):
        super().__init__()

        # Segmentation backbone (LinkNet + EfficientNet-B4)
        self.seg_model = smp.Linknet(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
        )

        # ── DYNAMIC channel detection (avoids hardcoding 1792 or 448) ──
        with torch.no_grad():
            dummy     = torch.zeros(1, 3, image_size, image_size)
            features  = self.seg_model.encoder(dummy)
            enc_ch    = features[-1].shape[1]   # actual bottleneck channels
        print(f'  Encoder bottleneck channels auto-detected: {enc_ch}')

        # Risk classification head
        self.risk_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(enc_ch, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

        # Depth regression head
        self.depth_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(enc_ch, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.ReLU(inplace=True),   # depth >= 0
        )

    def forward(self, x):
        features    = self.seg_model.encoder(x)
        bottleneck  = features[-1]
        water_mask  = self.seg_model(x)          # (B, 1, H, W) — segmentation
        risk_logits = self.risk_head(bottleneck) # (B, 5)
        depth_pred  = self.depth_head(bottleneck)# (B, 1)
        return water_mask, risk_logits, depth_pred


# Instantiate
model = FloodNetV2(num_classes=NUM_CLASSES, image_size=IMAGE_SIZE).to(DEVICE)

# ── Load V-FloodNet weights if download succeeded ──────────────────
if USE_VFLOODNET and pathlib.Path(PRETRAIN_PATH).exists():
    ckpt = torch.load(PRETRAIN_PATH, map_location='cpu')
    sd   = ckpt.get('model_state_dict', ckpt)   # handle both formats

    # Only load keys that exist in our model with matching shapes
    model_sd   = model.state_dict()
    compatible = {
        k: v for k, v in sd.items()
        if k in model_sd and model_sd[k].shape == v.shape
    }
    loaded, skipped = len(compatible), len(sd) - len(compatible)
    model.load_state_dict(compatible, strict=False)
    print(f'  V-FloodNet weights loaded: {loaded} layers (skipped {skipped})')
else:
    print('  Using ImageNet weights only')

# ── Partial freeze: freeze bottom 60%, train top 40% + heads ────────
encoder_layers = list(model.seg_model.encoder.parameters())
freeze_until   = int(len(encoder_layers) * 0.6)
for i, p in enumerate(encoder_layers):
    p.requires_grad = (i >= freeze_until)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'  Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
print('Model ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 8 — AUGMENTATIONS + DATASET
#  V-FloodNet augmentation pipeline
# ─────────────────────────────────────────────────────────────────

import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image

RISK2IDX = {c: i for i, c in enumerate(RISK_CLASSES)}

TRAIN_TF = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    # V-FloodNet augmentations
    A.ColorJitter(brightness=(0.6, 1.2), contrast=(0.6, 1.4),
                  saturation=0.3, hue=0.1, p=0.8),
    A.Affine(rotate=(-20, 20), translate_percent=0.2,
             scale=(0.7, 1.3), shear=(-20, 20), p=0.7),
    A.RandomResizedCrop(height=IMAGE_SIZE, width=IMAGE_SIZE,
                        scale=(0.6, 1.0), p=0.5),
    A.HorizontalFlip(p=0.5),
    A.GaussNoise(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

VAL_TF = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class FloodDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows = rows
        self.tf   = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row    = self.rows[idx]
        img    = np.array(Image.open(pathlib.Path(IMG_DIR) / row['filename']).convert('RGB'))
        img    = self.tf(image=img)['image']           # (3, H, W) float tensor
        risk   = RISK2IDX.get(row['risk_level'], 0)
        depth  = float(row['water_depth_cm']) / 100.0  # normalise to ~0-4
        return img, risk, depth

train_ds  = FloodDataset(train_rows, TRAIN_TF)
test_ds   = FloodDataset(test_rows,  VAL_TF)
train_dl  = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=2, pin_memory=(DEVICE=='cuda'))
test_dl   = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=2, pin_memory=(DEVICE=='cuda'))

print(f'Train batches: {len(train_dl)}  |  Test batches: {len(test_dl)}')
print('Dataset ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 9 — MULTI-TASK TRAINING LOOP
#  FIX 1: torch.amp.GradScaler (not deprecated torch.cuda.amp)
#  FIX 2: depth normalised by /100 before SmoothL1
# ─────────────────────────────────────────────────────────────────

import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Losses ────────────────────────────────────────────────────────
dice_loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)
ce_loss_fn   = nn.CrossEntropyLoss()
depth_loss_fn= nn.SmoothL1Loss()

optimizer  = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                   lr=LEARNING_RATE, weight_decay=1e-4)
scheduler  = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                patience=4)

# FIX: correct amp API for newer PyTorch (no deprecation warning)
use_amp = (DEVICE == 'cuda')
scaler  = torch.amp.GradScaler('cuda') if use_amp else None

history = {'train_loss': [], 'val_acc': [], 'val_mae': []}
best_acc, best_epoch = 0.0, 0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    for imgs, risks, depths in train_dl:
        imgs   = imgs.to(DEVICE)
        risks  = risks.to(DEVICE)
        depths = depths.float().to(DEVICE).unsqueeze(1)  # (B,1)

        optimizer.zero_grad()
        if use_amp:
            with torch.amp.autocast('cuda'):
                mask, risk_logits, depth_pred = model(imgs)
                # Proxy water mask: flood_detected from risk > 0
                water_proxy = (risks > 0).float().unsqueeze(-1).unsqueeze(-1)
                water_proxy = water_proxy.expand_as(mask)
                l_seg   = dice_loss_fn(mask, water_proxy)
                l_risk  = ce_loss_fn(risk_logits, risks)
                l_depth = depth_loss_fn(depth_pred, depths)
                loss    = W_SEG * l_seg + W_RISK * l_risk + W_DEPTH * l_depth
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            mask, risk_logits, depth_pred = model(imgs)
            water_proxy = (risks > 0).float().unsqueeze(-1).unsqueeze(-1).expand_as(mask)
            l_seg   = dice_loss_fn(mask, water_proxy)
            l_risk  = ce_loss_fn(risk_logits, risks)
            l_depth = depth_loss_fn(depth_pred, depths)
            loss    = W_SEG * l_seg + W_RISK * l_risk + W_DEPTH * l_depth
            loss.backward()
            optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dl)
    scheduler.step(avg_loss)

    # ── Validate ───────────────────────────────────────────────────
    model.eval()
    correct, total, mae_sum = 0, 0, 0.0
    with torch.no_grad():
        for imgs, risks, depths in test_dl:
            imgs   = imgs.to(DEVICE)
            risks  = risks.to(DEVICE)
            depths = depths.float().to(DEVICE)

            _, risk_logits, depth_pred = model(imgs)
            preds    = risk_logits.argmax(dim=1)
            correct += (preds == risks).sum().item()
            total   += risks.size(0)
            # FIX: squeeze(1) to avoid scalar collapse at batch_size=1
            mae_sum += (depth_pred.squeeze(1) * 100 - depths * 100).abs().sum().item()

    val_acc = correct / max(total, 1)
    val_mae = mae_sum  / max(total, 1)
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    history['val_mae'].append(val_mae)

    if val_acc > best_acc:
        best_acc, best_epoch = val_acc, epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_mae': val_mae,
        }, f'{CKPT_DIR}/best_floodnet_v2.pth')

    if epoch % 5 == 0 or epoch == EPOCHS:
        print(f'Epoch {epoch:3d}/{EPOCHS} | loss {avg_loss:.4f} | '
              f'val_acc {val_acc:.3f} | val_MAE {val_mae:.1f}cm '
              f'{"  ← best" if epoch == best_epoch else ""}')

# ── Plot training curves ───────────────────────────────────────────
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
ax1.plot(history['train_loss']); ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch')
ax2.plot(history['val_acc']);    ax2.set_title('Val Accuracy'); ax2.set_xlabel('Epoch')
ax3.plot(history['val_mae']);    ax3.set_title('Val MAE (cm)'); ax3.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/charts/training_curves.png', dpi=120)
plt.show()

print(f'\nTraining done. Best epoch {best_epoch} — accuracy {best_acc:.3f}')
print(f'Checkpoint saved: {CKPT_DIR}/best_floodnet_v2.pth')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 10 — EVALUATE ON TEST SET
#  FIX: depth_pred.squeeze(1) not squeeze() — prevents scalar at B=1
# ─────────────────────────────────────────────────────────────────

from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score
)
import numpy as np
import matplotlib.pyplot as plt

# Load best checkpoint
ckpt = torch.load(f'{CKPT_DIR}/best_floodnet_v2.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val_acc {ckpt['val_acc']:.3f})")

all_preds, all_labels, all_depths_pred, all_depths_true = [], [], [], []

with torch.no_grad():
    for imgs, risks, depths in test_dl:
        imgs  = imgs.to(DEVICE)
        _, risk_logits, depth_pred = model(imgs)
        preds = risk_logits.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(risks.tolist())
        # FIX: squeeze(1) keeps shape (B,) even when B=1
        all_depths_pred.extend((depth_pred.squeeze(1).cpu() * 100).tolist())
        all_depths_true.extend((depths * 100).tolist())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_depths_pred = np.array(all_depths_pred)
all_depths_true = np.array(all_depths_true)

acc    = (all_preds == all_labels).mean()
mae    = np.abs(all_depths_pred - all_depths_true).mean()
f1     = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
prec   = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
rec    = recall_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f'\n=== Test Results ===');
print(f'  Accuracy : {acc:.4f}  (target ≥ 0.80)')
print(f'  F1 score : {f1:.4f}  (target ≥ 0.85)')
print(f'  Precision: {prec:.4f}')
print(f'  Recall   : {rec:.4f}')
print(f'  Depth MAE: {mae:.2f} cm  (target ≤ 8 cm)')

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(RISK_CLASSES, rotation=45, ha='right')
ax.set_yticklabels(RISK_CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() * 0.5 else 'black')
plt.colorbar(im)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/charts/confusion_matrix.png', dpi=120)
plt.show()

print('\nPer-class report:')
print(classification_report(all_labels, all_preds,
                             target_names=RISK_CLASSES, zero_division=0))

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 11 — DEEP METRICS  (optional — set False to skip)
#  Depends on Cell 10 variables: all_preds, all_labels,
#  all_depths_pred, all_depths_true, f1, prec, rec, acc, mae
# ─────────────────────────────────────────────────────────────────

RUN_DEEP_METRICS = True

if RUN_DEEP_METRICS:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve
    from sklearn.preprocessing import label_binarize

    # 1. Per-class detailed report (already printed above, saved to file here)
    report_str = classification_report(
        all_labels, all_preds,
        target_names=RISK_CLASSES, zero_division=0
    )
    with open(f'{OUT_DIR}/metrics/per_class_report.txt', 'w') as f:
        f.write(report_str)

    # 2. Depth error histogram
    errors = all_depths_pred - all_depths_true
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.hist(errors, bins=40, color='steelblue', edgecolor='white')
    ax1.axvline(0, color='red', linestyle='--')
    ax1.set_xlabel('Depth error (cm)'); ax1.set_ylabel('Count')
    ax1.set_title('Depth Prediction Error Distribution')

    ax2.scatter(all_depths_true, all_depths_pred, alpha=0.4, s=15)
    lim = max(all_depths_true.max(), all_depths_pred.max())
    ax2.plot([0, lim], [0, lim], 'r--')
    ax2.set_xlabel('Actual depth (cm)'); ax2.set_ylabel('Predicted depth (cm)')
    ax2.set_title('Predicted vs Actual Depth')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/metrics/depth_errors.png', dpi=120)
    plt.show()

    # 3. ROC curves (one-vs-rest for each class)
    y_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))

    # We need probability scores for ROC — re-run inference to collect softmax
    model.eval()
    all_probs = []
    with torch.no_grad():
        for imgs, _, _ in test_dl:
            imgs = imgs.to(DEVICE)
            _, logits, _ = model(imgs)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
    all_probs = np.vstack(all_probs)

    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ['navy','crimson','darkorange','forestgreen','purple']
    for i, (cls, col) in enumerate(zip(RISK_CLASSES, colors)):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=col, lw=1.5, label=f'{cls} (AUC={roc_auc:.2f})')
    ax.plot([0,1],[0,1],'k--'); ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC Curves')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/metrics/roc_curves.png', dpi=120)
    plt.show()

    # 4. Full scorecard (pass / fail)
    targets = [
        ('Accuracy',       acc,  0.80, '>='),
        ('F1 (weighted)',  f1,   0.85, '>='),
        ('Depth MAE (cm)', mae,  8.0,  '<='),
        ('Precision',      prec, 0.80, '>='),
        ('Recall',         rec,  0.80, '>='),
    ]
    print('\n=== Model Scorecard ===')
    all_pass = True
    for name, val, thresh, op in targets:
        ok = (val >= thresh) if op == '>=' else (val <= thresh)
        status = '✅ PASS' if ok else '❌ FAIL'
        if not ok: all_pass = False
        print(f'  {status}  {name}: {val:.4f}  (target {op} {thresh})')
    print(f'\n  Overall: {"✅ ALL TARGETS MET" if all_pass else "❌ SOME TARGETS MISSED"}')

    # Save scorecard
    with open(f'{OUT_DIR}/metrics/scorecard.txt', 'w') as f:
        for name, val, thresh, op in targets:
            ok = (val >= thresh) if op == '>=' else (val <= thresh)
            f.write(f'{"PASS" if ok else "FAIL"}  {name}: {val:.4f}  (target {op} {thresh})\n')
else:
    print('Deep metrics skipped (set RUN_DEEP_METRICS = True to enable)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 12 — DOWNLOAD RESULTS ZIP
#  Packages: best checkpoint + charts + metrics + labels
# ─────────────────────────────────────────────────────────────────

import zipfile, pathlib
from google.colab import files

ZIP_PATH = '/content/floodnet_v2_results.zip'

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Best model checkpoint
    ckpt_path = pathlib.Path(f'{CKPT_DIR}/best_floodnet_v2.pth')
    if ckpt_path.exists():
        zf.write(ckpt_path, 'checkpoints/best_floodnet_v2.pth')

    # Charts
    for p in pathlib.Path(f'{OUT_DIR}/charts').glob('*.png'):
        zf.write(p, f'charts/{p.name}')

    # Metrics (if deep metrics ran)
    for p in pathlib.Path(f'{OUT_DIR}/metrics').glob('*'):
        zf.write(p, f'metrics/{p.name}')

    # Labels CSV
    labels_path = pathlib.Path(LABEL_PATH)
    if labels_path.exists():
        zf.write(labels_path, 'labels/labels.csv')

size_mb = pathlib.Path(ZIP_PATH).stat().st_size / 1e6
print(f'ZIP ready: {size_mb:.1f} MB → {ZIP_PATH}')
print('Downloading...')
files.download(ZIP_PATH)
print('Done')